# Root-MUSIC
## Tutorial 8: Polynomial Rooting for DOA Estimation

Root-MUSIC (Barabell, 1983) avoids the spectral peak search in MUSIC by reformulating DOA estimation as a **polynomial rooting problem**.  This yields:
- No grid-resolution limitation
- Computational savings (no angle sweep)
- Directly computable roots via eigenvalues

Topics:
1. **Polynomial formulation** of MUSIC
2. **Root finding and root selection**
3. **Accuracy comparison with spectral MUSIC**
4. **Sensitivity to grid density in spectral MUSIC**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.subspace import MUSIC, RootMUSIC

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array = UniformLinearArray(M=M, d=0.5)
sm    = SignalModel(array)
music = MUSIC(array)
root_music = RootMUSIC(array)

doas_true = np.deg2rad([-20.0, 15.0])
K = len(doas_true)
snr_db = 10
N = 200

print("Setup complete.")

## 1. Polynomial Formulation

The MUSIC pseudo-spectrum denominator is:

$$f(\theta) = \mathbf{a}^H(\theta)\mathbf{C}\mathbf{a}(\theta), \quad \mathbf{C} = \mathbf{U}_n\mathbf{U}_n^H$$

For a ULA with $\mathbf{a}(\theta) = [1, z, z^2, \ldots, z^{M-1}]^T$ where $z = e^{-j2\pi d\sin\theta}$, this becomes a polynomial in $z$:

$$p(z) = \sum_{m=-(M-1)}^{M-1} c_m z^{-m}, \quad c_m = \sum_{i} [\mathbf{C}]_{i, i+m}$$

The polynomial has degree $2(M-1)$.  True DOAs correspond to roots **on or near the unit circle**.

**Root selection**: of the $2(M-1)$ roots, take the $K$ roots inside (or closest to) the unit circle.  Their phases encode the DOAs.

In [ ]:
X, _, _ = sm.generate_signals(doas_true, N, snr_db, seed=42)
R = X @ X.conj().T / N
eigenvals, eigenvecs = np.linalg.eigh(R)
eigenvals = eigenvals[::-1]; eigenvecs = eigenvecs[:, ::-1]
U_n = eigenvecs[:, K:]

# Build polynomial coefficient matrix C = U_n U_n^H
C = U_n @ U_n.conj().T

# Polynomial coefficients: sum the diagonals of C
# Coefficient c_m = trace of the m-th diagonal (0 = main, +m = upper, -m = lower)
# Polynomial: sum_{l=0}^{2M-2} coeff[l] z^l  where z = e^{-j*omega}
coeffs = np.zeros(2*M - 1, dtype=complex)
for i in range(-(M-1), M):
    coeffs[i + M - 1] = np.sum(np.diag(C, i))

# The polynomial in z
roots = np.roots(coeffs[::-1])  # np.roots expects highest power first

# Visualise roots on unit circle
fig, ax = plt.subplots(figsize=(8, 8))
theta_circle = np.linspace(0, 2*np.pi, 500)
ax.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', lw=1, alpha=0.5,
        label='Unit circle')
ax.plot(roots.real, roots.imag, 'bx', ms=8, lw=2, label='Polynomial roots')

# Mark roots closest to unit circle (the signal roots)
dist_to_circle = np.abs(np.abs(roots) - 1)
signal_root_idx = np.argsort(dist_to_circle)[:K]
ax.plot(roots[signal_root_idx].real, roots[signal_root_idx].imag,
        'ro', ms=10, label=f'Selected {K} signal roots')

for th in doas_true:
    z_true = np.exp(-1j * 2*np.pi * 0.5 * np.sin(th))
    ax.plot(z_true.real, z_true.imag, 'g^', ms=12)

ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_xlabel('Real'); ax.set_ylabel('Imaginary')
ax.set_title('Root-MUSIC: Polynomial Roots in the Complex Plane')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Estimate DOAs from selected roots
signal_roots = roots[signal_root_idx]
omega_est = -np.angle(signal_roots)           # ω = 2π d sin θ
sin_theta_est = omega_est / (2*np.pi * 0.5)
sin_theta_est = np.clip(sin_theta_est, -1, 1)
doas_root = np.sort(np.arcsin(sin_theta_est))
print(f"True DOAs   : {np.round(np.rad2deg(doas_true), 4)} °")
print(f"Root-MUSIC  : {np.round(np.rad2deg(doas_root), 4)} °")

## 2. Using the RootMUSIC Class

In [ ]:
doas_rmu = root_music.estimate(X, K=K)
print(f"True DOAs         : {np.round(np.rad2deg(doas_true), 4)} °")
print(f"Root-MUSIC (class): {np.round(np.rad2deg(doas_rmu), 4)} °")

## 3. Grid Resolution in Spectral MUSIC

Spectral MUSIC is limited by the angular grid spacing.  Root-MUSIC has **no such limitation**.

In [ ]:
grid_sizes = [37, 91, 181, 721, 3601]   # number of grid points

fig, ax = plt.subplots(figsize=(12, 6))
X_grid, _, _ = sm.generate_signals(doas_true, N=500, snr_db=15, seed=0)

for ng in grid_sizes:
    ag = np.linspace(-np.pi/2, np.pi/2, ng)
    est = music.estimate(X_grid, K=K, angle_grid=ag)
    err = np.max(np.abs(np.rad2deg(est) - np.rad2deg(np.sort(doas_true))))
    print(f"Grid points = {ng:5d}  →  max error = {err:.4f}°")

# Root-MUSIC has no grid
est_rmu = root_music.estimate(X_grid, K=K)
err_rmu = np.max(np.abs(np.rad2deg(est_rmu) - np.rad2deg(np.sort(doas_true))))
print(f"Root-MUSIC          →  max error = {err_rmu:.4f}° (no grid)")

## 4. Statistical Comparison: Spectral vs Root-MUSIC

In [ ]:
angle_grid_fine = np.linspace(-np.pi/2, np.pi/2, 3601)
n_trials = 300
snr_range_cmp = np.arange(-5, 26, 5)

def rmse_method(method_fn, doas, N_, snr_, trials):
    errs = []
    for t in range(trials):
        X_, _, _ = sm.generate_signals(doas, N_, snr_, seed=t)
        try:
            est = np.sort(method_fn(X_))
            errs.append(np.sqrt(np.mean((est - np.sort(doas))**2)))
        except Exception:
            errs.append(np.pi)
    return np.rad2deg(np.sqrt(np.mean(np.array(errs)**2)))

rmse_spec = [rmse_method(lambda X_: music.estimate(X_, K=K, angle_grid=angle_grid_fine),
                         doas_true, N, s, n_trials) for s in snr_range_cmp]
rmse_root = [rmse_method(lambda X_: root_music.estimate(X_, K=K),
                         doas_true, N, s, n_trials) for s in snr_range_cmp]

fig, ax = plt.subplots(figsize=(11, 6))
ax.semilogy(snr_range_cmp, rmse_spec, 'b-o', ms=6, lw=2, label='Spectral MUSIC (3601 pts)')
ax.semilogy(snr_range_cmp, rmse_root, 'r-s', ms=6, lw=2, label='Root-MUSIC')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('RMSE (°)')
ax.set_title(f'Spectral vs Root-MUSIC RMSE  (M={M}, N={N})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

- Root-MUSIC converts DOA estimation to polynomial rooting, **eliminating grid-resolution errors**.
- Signal roots lie on (or very close to) the unit circle; their phase gives the DOA.
- Root-MUSIC typically achieves slightly **lower RMSE** than spectral MUSIC because it is not limited by grid spacing.
- Computational cost is $O(M^3)$ for the eigendecomposition + $O(M^2)$ for polynomial root finding — comparable to spectral MUSIC.

## Exercises
1. Derive the polynomial coefficient $c_m$ in terms of the elements $[\mathbf{C}]_{ij}$ of the noise projection matrix.
2. For $K=1$, show that the two signal roots are complex conjugates of each other and that one lies inside and one outside the unit circle.
3. Implement a refinement: use the Root-MUSIC estimate as an initial point and run a 1D Newton–Raphson step on $p(z)$ to refine the root estimate.